In [1]:
import Pkg
Pkg.activate("../chebcoefs")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D/chebcoefs`


In [2]:
Pkg.status()

Status `~/repos/WaveGreen2D/chebcoefs/Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
⌃ [13f3f980] CairoMakie v0.15.10
  [cf66c380] FastChebInterp v1.3.1
  [033835bb] JLD2 v0.6.4
  [1fd47b50] QuadGK v2.11.3
  [90137ffa] StaticArrays v1.9.18
Info Packages marked with ⌃ have new versions available and may be upgradable.


In [3]:
using BenchmarkTools
using CairoMakie
using JLD2
using QuadGK
using Random

In [4]:
using CairoMakie

[ Info: Precompiling CairoMakie [13f3f980-e62b-5c42-98c6-ff1f3baf88f0](cache misses: wrong dep version loaded (2))
[ Info: Precompiling CairoMakie [13f3f980-e62b-5c42-98c6-ff1f3baf88f0] (cache misses: wrong dep version loaded (4))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

[177114] signal 2: Interrupt
in expression starting at /home/rodpcastro/.julia/packages/CairoMakie/hql6v/src/CairoMakie.jl:3
unknown function (ip: 0x793f064acae2) at /usr/lib/x86_64-linux-gnu/libc.so.6
unknown function (ip: 0x793f064a067b) at /usr/lib/x86_64-linux-gnu/libc.so.6
epoll_pwait at /usr/lib/x86_64-linux-gnu/libc.so.6 (unknown line)

[177121] signal 2:

LoadError: InterruptException:


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [ ]:
# The objective of this script is to study the optimal quadrature order for L₁ and L₂.
# From the results, we conclude that the optimal quadrature order is between 24 and 34.

cheb_dir = joinpath(dirname(@__DIR__), "chebcoefs")

data_dir = joinpath(cheb_dir, "data")
img_dir = joinpath(cheb_dir, "images")
bench_file = joinpath(data_dir, "qorder_benchmarks.jld2")
img_file = joinpath(img_dir, "qorder_benchmarks_plots.svg")

Random.seed!(18)
tol = eps()
imax = 1e4


function L₁(x::AbstractVector{<:Real}; qorder::Int=7)
    A, B, C = x
    H = 10.0^C  # C = log₁₀H

    f(u) = (u + H) / (u - H - (u + H) * exp(-2u))
    g(u) = exp(-u * (2 + B)) + exp(-u * (2 - B))
    h(u) = cos(u * A)
    p(u) = (f(u) * g(u) * h(u) + exp(-u)) / u

    path = (0.0, H + im, H + 1.0, Inf)
    y = 0.0

    for i in 1:length(path)-1
        y += quadgk(
            p, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder
        )[1]
    end

    return real(y)
end


function L₂(x::AbstractVector{<:Real}; qorder::Int=7)
    A, B, C = x
    H = 10.0^C  # C = log₁₀H

    f(u) = (u + H) / (u - H - (u + H) * exp(-2u))
    g(u) = (u + H)^2 / ((u - H)^2 - (u^2 - H^2) * exp(-2u))
    p(u) = exp(-u * (2 + B))
    q(u) = exp(-u * (4 - B))
    r(u) = cos(u * A)

    h(u) = (f(u) * p(u) + g(u) * q(u)) * r(u) / u

    path = (0.0, H + im, H + 1.0, Inf)
    y = 0.0

    for i in 1:length(path)-1
        y += quadgk(
            h, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder
        )[1]
    end

    return real(y)
end


if !isfile(bench_file)
    error("There is no benchmark file to load.")
end

println("Loading benchmark")
@load bench_file qorder_vals L₁_bench L₂_bench

println("Plotting")
mkpath(img_dir)


L₁_time = [bench.time for bench in values(L₁_bench)]
L₁_memory = [bench.memory for bench in values(L₁_bench)]
L₁_allocs = [bench.allocs for bench in values(L₁_bench)]

L₂_time = [bench.time for bench in values(L₂_bench)]
L₂_memory = [bench.memory for bench in values(L₂_bench)]
L₂_allocs = [bench.allocs for bench in values(L₂_bench)]


fig = Figure(size=(1200, 400))
axes = [Axis(fig[1, m], xlabel="Quadrature order") for m in 1:3]

axes[1].title = "Mean execution time x Quadrature order"
lines!(axes[1], qorder_vals, L₁_time, color=:green, linestyle=:solid, label=L"L_1")
lines!(axes[1], qorder_vals, L₂_time, color=:red, linestyle=:dash, label=L"L_2")
axislegend(axes[1])

axes[2].title = "Allocated memory x Quadrature order"
lines!(axes[2], qorder_vals, L₁_memory, color=:green, linestyle=:solid, label=L"L_1")
lines!(axes[2], qorder_vals, L₂_memory, color=:red, linestyle=:dash, label=L"L_2")

axes[3].title = "Number of allocations x Quadrature order"
lines!(axes[3], qorder_vals, L₁_allocs, color=:green, linestyle=:solid, label=L"L_1")
lines!(axes[3], qorder_vals, L₂_allocs, color=:red, linestyle=:dash, label=L"L_2")

save(img_file, fig)
println("Done")

fig